# Whisper LoRA 微调与测试 (衡东方言)

使用 PEFT/LoRA 微调 OpenAI Whisper-base 模型，支持中文方言识别。

基于 fine-tuning-expert skill 最佳实践配置 LoRA 参数。

## 1. 环境检查

In [ ]:
import torch
import subprocess
import json
import os

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

try:
    import transformers
    print(f"transformers: {transformers.__version__}")
except ImportError:
    print("transformers 未安装")

try:
    import peft
    print(f"peft: {peft.__version__}")
except ImportError:
    print("peft 未安装")

try:
    import librosa
    print(f"librosa: {librosa.__version__}")
except ImportError:
    print("librosa 未安装")

try:
    import datasets
    print(f"datasets: {datasets.__version__}")
except ImportError:
    print("datasets 未安装")

## 2. 数据检查

In [ ]:
TRAIN_JSONL = "/mnt/data/prepared_data/whisper/train.jsonl"
VAL_JSONL = "/mnt/data/prepared_data/whisper/val.jsonl"

for path, label in [(TRAIN_JSONL, '训练集'), (VAL_JSONL, '验证集')]:
    if os.path.exists(path):
        count = sum(1 for _ in open(path))
        print(f"  {label}: {count} 条")
    else:
        print(f"  {label}: 不存在! {path}")

# 音频加载测试
if os.path.exists(TRAIN_JSONL):
    with open(TRAIN_JSONL) as f:
        sample = json.loads(f.readline())
    audio = sample.get('audio_filepath') or sample.get('source')
    text = sample.get('text') or sample.get('target')
    print(f"\n数据样例:")
    print(f"  音频: {audio}")
    print(f"  文本: {text}")
    print(f"  字段: {list(sample.keys())}")

## 3. 准备数据集

In [ ]:
from datasets import Dataset

def load_jsonl(path):
    """加载 JSONL 数据集"""
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            audio = item.get('audio_filepath') or item.get('source')
            text = item.get('text') or item.get('target')
            if audio and text:
                data.append({'audio': audio, 'text': text})
    return data

train_data = load_jsonl(TRAIN_JSONL)
val_data = load_jsonl(VAL_JSONL)

print(f"训练集: {len(train_data)} 条")
print(f"验证集: {len(val_data)} 条")

# 创建 HuggingFace Dataset
train_ds = Dataset.from_list(train_data)
val_ds = Dataset.from_list(val_data)
print(f"\nDataset 创建成功!")
print(f"train_ds: {train_ds}")
print(f"val_ds: {val_ds}")

## 4. 加载模型和处理器

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model
import torch
import time

# 模型路径（离线加载，避免网络请求）
MODEL_ID = "/mnt/data"
LANGUAGE = "chinese"
TASK = "transcribe"

print(f"加载 Whisper-base from {MODEL_ID}...")
start = time.time()

# 加载处理器
processor = WhisperProcessor.from_pretrained(
    MODEL_ID,
    local_files_only=True,
    language=LANGUAGE,
    task=TASK
)

# 加载基座模型
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID,
    local_files_only=True,
)
model = model.to("cuda")
model.eval()

print(f"模型加载完成, 耗时: {time.time() - start:.1f}s")

## 5. 配置 LoRA (基于 fine-tuning-expert 最佳实践)

In [ ]:
# LoRA 配置
# Whisper-base (139M 参数) 使用中等 rank 即可
lora_config = LoraConfig(
    r=16,                          # Rank - 8-16 适合 ASR 任务
    lora_alpha=32,                  # Alpha - 通常 2× rank
    lora_dropout=0.1,              # Dropout 防止过拟合
    target_modules=[
        "q_proj", "v_proj",       # Attention q, v
    ],
    bias="none",                   # 不训练 bias
    task_type=None,                # Whisper 使用 None
)

# 应用 LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("\nLoRA 配置:")
print(f"  rank: {lora_config.r}")
print(f"  alpha: {lora_config.lora_alpha}")
print(f"  target_modules: {lora_config.target_modules}")

## 6. 数据预处理

In [ ]:
import librosa
import numpy as np

def prepare_dataset(batch):
    """预处理单个样本"""
    audio_path = batch["audio"]
    
    # 加载音频
    array, sr = librosa.load(audio_path, sr=16000)
    
    # 处理音频特征
    input_features = processor.feature_extractor(
        raw_speech=array,
        sampling_rate=sr,
        return_tensors="pt",
    ).input_features
    
    # 编码文本
    labels = processor.tokenizer(
        batch["text"],
        return_tensors="pt",
    ).input_ids
    
    batch["input_features"] = input_features[0]
    batch["labels"] = labels[0]
    return batch

print("测试预处理...")
test_sample = train_data[0]
test_result = prepare_dataset(test_sample)
print(f"input_features shape: {test_result['input_features'].shape}")
print(f"labels shape: {test_result['labels'].shape}")

# 预处理全部数据
print("\n预处理训练集...")
train_ds = train_ds.map(prepare_dataset, remove_columns=train_ds.column_names)
print(f"训练集预处理完成: {len(train_ds)}")

print("预处理验证集...")
val_ds = val_ds.map(prepare_dataset, remove_columns=val_ds.column_names)
print(f"验证集预处理完成: {len(val_ds)}")

## 7. 训练配置

In [ ]:
from transformers import TrainingArguments, Trainer
import torch
import time

OUTPUT_DIR = "/mnt/output/whisper_lora"
os.makedirs(OUTPUT_DIR, exist_ok=True)


class WhisperDataCollator:
    """Whisper 专用数据整理器，处理 input_features 和 labels"""

    def __call__(self, features):
        # input_features 已经是固定长度的 numpy array，直接转 tensor stack
        input_features = [torch.tensor(f["input_features"], dtype=torch.float32) for f in features]
        batch = {"input_features": torch.stack(input_features)}

        # labels 转 tensor，再动态 padding
        labels = [torch.tensor(f["labels"], dtype=torch.long) for f in features]
        max_len = max(l.size(0) for l in labels)
        padded = []
        for l in labels:
            pad_len = max_len - l.size(0)
            if pad_len > 0:
                l = torch.cat([l, torch.full((pad_len,), -100, dtype=torch.long)])
            padded.append(l)
        batch["labels"] = torch.stack(padded)

        return batch


data_collator = WhisperDataCollator()

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    max_steps=2000,
    logging_steps=50,
    save_steps=500,
    eval_strategy="steps",
    eval_steps=500,
    save_total_limit=3,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    save_only_model=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    dataloader_num_workers=4,
    remove_unused_columns=False,
)

## 8. 开始训练

In [ ]:
print("开始训练...")
print("=" * 60)
start = time.time()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    processing_class=processor,
)

trainer.train()

print(f"\n{'=' * 60}")
print(f"训练完成! 耗时: {time.time() - start:.1f}s")
print(f"模型保存在: {OUTPUT_DIR}")

## 9. 保存 LoRA 权重

In [ ]:
LORA_OUTPUT = os.path.join(OUTPUT_DIR, "lora_adapter")
model.save_pretrained(LORA_OUTPUT)
processor.save_pretrained(LORA_OUTPUT)
print(f"LoRA 权重已保存: {LORA_OUTPUT}")

## 10. 加载微调模型并测试

In [ ]:
from peft import PeftModel

print("加载 LoRA 权重...")
model_load = WhisperForConditionalGeneration.from_pretrained("/mnt/data", local_files_only=True)
model_load = PeftModel.from_pretrained(model_load, LORA_OUTPUT)
model_load = model_load.to("cuda")
model_load.eval()

## 11. 清理内存

In [ ]:
del model, model_load, trainer
torch.cuda.empty_cache()
import gc
gc.collect()
print("内存已清理")